<a href="https://colab.research.google.com/github/Innovatewithapple/YoutubeSummarizationNLP/blob/main/YoutubeSummaryNLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yt-dlp openai-whisper

In [ ]:
import numpy as np
import random
import os
import torch

def setProductionSeed(isActive,seed=42):
  if isActive == True:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
      torch.cuda.manual_seed(seed)
      torch.cuda.manual_seed_all(seed)

      torch.backends.cudnn.deterministic = True
      torch.backends.cudnn.benchmark = False

setProductionSeed(True)

In [ ]:
import yt_dlp
import whisper
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer,AutoModelForCausalLM
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab')
from torch.utils.data import DataLoader,Dataset

In [ ]:
from huggingface_hub import login
login(token="")

In [ ]:
#-----------Transcribe Model-----------!
transcribe_Model = whisper.load_model('small')

#--------Decoder-------!
decoder_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", trust_remote_code=True)
decoder_tokenizer.pad_token = decoder_tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct",dtype=torch.float16,device_map="auto",trust_remote_code=True)

In [ ]:
def download_audio(youtube_url, output_path="audio"):
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': f'{output_path}/%(title)s.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        filename = ydl.prepare_filename(info)
        # Return the actual .mp3 path
        return filename.rsplit('.', 1)[0] + '.mp3'

In [ ]:
audio_path = download_audio("https://www.youtube.com/watch?v=RRKwmeyIc24")
print(f"Audio saved to: {audio_path}")

In [ ]:
#--------Transcribe Audio----------!
transcribe_result = transcribe_Model.transcribe(audio_path)
transcript = transcribe_result['text']
print(f'Transcript: \n{transcript}')

In [ ]:
def Chunk_process(text,token_size,overlap):
  sentences = sent_tokenize(text)
  sent_token_counts = [len(decoder_tokenizer.encode(sent,add_special_tokens=False)) for sent in sentences]

  chunks = []
  current_chunk = []
  current_len = 0

  for sent,token_len in zip(sentences,sent_token_counts):
    if current_len + token_len <= token_size:
      current_chunk.append(sent) # because sent itself a senetence not character so no need to use " "
      current_len += token_len
    else:
      if current_chunk:
        chunks.append(" ".join(current_chunk))
      current_chunk = current_chunk[-overlap:] + [sent]
      current_len = token_len

  if current_chunk:
    chunks.append(" ".join(current_chunk))

  return chunks

In [ ]:
all_chunks = Chunk_process(transcript,token_size=400,overlap=1)
print(all_chunks[0])

In [ ]:
train_Data = decoder_tokenizer(text=all_chunks,padding=True,max_length=512,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [ ]:
class GetInsideInfo(Dataset):
  def __init__(self,encoding):
    self.encoding = encoding

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    input_ids = self.encoding['input_ids'][idx]
    attention_mask = self.encoding['attention_mask'][idx]

    return {
        'input_ids':input_ids,
        'attention_mask':attention_mask,
    }

In [ ]:
train_ds = GetInsideInfo(train_Data)
train_loader = DataLoader(dataset=train_ds,batch_size=32,shuffle=True,pin_memory=True,num_workers=0)